# Training: Gender (Female / Male)

**Target mapping:** `female = 0`, `male = 1`. `ImageFolder` assigns class indices alphabetically, so the class folders must be named `female` and `male` consistently in every split. This notebook validates the mapping before training.

**Data:** real face images only (`split_gender_norm`). Fake images are not used, because their gender is a placeholder (`unknown`), not a real label. In the app, images judged fake are shown with gender "Unknown".

**Models:** CNN, ResNet50, VGG16, EfficientNet-B0, and ViT-B/16. Each model has its own training, save, evaluation, confusion-matrix, and ROC-curve cells


#### Import the required libraries:

In [66]:
import copy
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, roc_curve, classification_report, auc)
import plotly.express as px
import plotly.graph_objects as go


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce GTX 1650 Ti


### Load the dataset

`ImageFolder` treats each class subfolder as a target label and assigns indices in alphabetical order. With `female` and `male` folders, the expected mapping is `female = 0` and `male = 1`.


In [23]:
# Set the project directory.
PROJECT_DIR = Path(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final")

# Set the data and model directories.
DATA_DIR = PROJECT_DIR / "data"
TASK = "gender"

SPLIT_DIR = DATA_DIR / "split_gender"

# Save trained models in the existing project-level models directory.
MODEL_DIR = PROJECT_DIR / "models" / "gender"
MODEL_DIR.mkdir(parents=True, exist_ok=True)


CHARTS_DIR = PROJECT_DIR / "charts" / "gender"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

# Define the training, validation, and test directories.
TRAIN_DIR = SPLIT_DIR / "train"
VAL_DIR = SPLIT_DIR / "val"
TEST_DIR = SPLIT_DIR / "test"

# Check that all split directories exist.
split_dirs = [TRAIN_DIR, VAL_DIR, TEST_DIR]
missing_splits = [
    str(folder) for folder in split_dirs
    if not folder.is_dir()
]

if missing_splits:
    raise FileNotFoundError(
        "The following split directories are missing: "
        + ", ".join(missing_splits)
        + ". Run 03_data_split_train_val_test_split.ipynb first."
    )

# Class folders expected in every split (alphabetical order = ImageFolder index order).
EXPECTED_CLASSES = ["female", "male", "unknown"]

for split_dir in split_dirs:
    found_classes = sorted(folder.name for folder in split_dir.iterdir() if folder.is_dir())
    if found_classes != EXPECTED_CLASSES:
        raise FileNotFoundError(
            f"Class folders in {split_dir} are {found_classes}, expected {EXPECTED_CLASSES}"
        )

# Set training parameters.
BATCH_SIZE = 32  # Reduce to 16 if a CUDA out-of-memory error occurs.
PRETRAINED = True  # Set to False to train without pretrained ImageNet weights.
SEED = 42

# Set random seeds for reproducibility.
torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Data directory:", DATA_DIR)
print("Split directory:", SPLIT_DIR)
print("Model save directory:", MODEL_DIR)
print("Charts save directory:", CHARTS_DIR)
print("Training, validation, and test folders are ready.")

Data directory: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data
Split directory: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\split_gender
Model save directory: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\gender
Charts save directory: C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\charts\gender
Training, validation, and test folders are ready.


# Data Preprocessing:

### Transform Compose:

Red channel: mean = 0.485, std = 0.229

Green channel: mean = 0.456, std = 0.224

Blue channel: mean = 0.406, std = 0.225

### Train:

In [24]:
train_transform = transforms.Compose([
                                        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
                                        transforms.RandomHorizontalFlip(p=0.5),
                                        transforms.RandomRotation(degrees=10),
                                        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
                                        transforms.ToTensor(),
                                        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                             std=[0.229, 0.224, 0.225])
                                    ])

### Test/Val:

In [25]:
val_test_transform = transforms.Compose([
                                transforms.Resize((224, 224)),
                                transforms.ToTensor(),
                                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                     std=[0.229, 0.224, 0.225])
                                        ])

In [26]:
# Create datasets for training, training-set evaluation, validation, and testing.
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
train_eval_dataset = datasets.ImageFolder(TRAIN_DIR, transform=val_test_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_test_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=val_test_transform)

# Confirm that every split uses the same target-to-index mapping.
expected_class_to_idx = {"female": 0, "male": 1, "unknown": 2}
def normalize_class_mapping(class_to_idx):
    return {name.lower(): index for name, index in class_to_idx.items()}

train_mapping = normalize_class_mapping(train_dataset.class_to_idx)
val_mapping = normalize_class_mapping(val_dataset.class_to_idx)
test_mapping = normalize_class_mapping(test_dataset.class_to_idx)

for split_name, mapping in [
    ("training", train_mapping),
    ("validation", val_mapping),
    ("test", test_mapping)
]:
    if mapping != expected_class_to_idx:
        raise ValueError(
            f"Incorrect target mapping in the {split_name} split: {mapping}. "
            f"Expected {expected_class_to_idx}. Check the class folder names."
        )

class_names = train_dataset.classes
num_classes = len(class_names)

if num_classes != 3:
    raise ValueError(f"Expected exactly two classes (female and male), but found: {class_names}")

print("Training class mapping:", train_dataset.class_to_idx)
print("Validation class mapping:", val_dataset.class_to_idx)
print("Test class mapping:", test_dataset.class_to_idx)
print("Target mapping verified:", expected_class_to_idx)

image, label = train_dataset[0]
print("Example target index:", label, "->", class_names[label])
print("Image type:", type(image))
print("Image shape:", image.shape)


Training class mapping: {'female': 0, 'male': 1, 'unknown': 2}
Validation class mapping: {'female': 0, 'male': 1, 'unknown': 2}
Test class mapping: {'female': 0, 'male': 1, 'unknown': 2}
Target mapping verified: {'female': 0, 'male': 1, 'unknown': 2}
Example target index: 0 -> female
Image type: <class 'torch.Tensor'>
Image shape: torch.Size([3, 224, 224])


In [27]:
# Accuracy that a model would get by always predicting the biggest class (test set).
print("Predict-biggest-class baseline (test):", round(np.bincount(test_dataset.targets).max() / len(test_dataset), 3))


Predict-biggest-class baseline (test): 0.511


# Data Loader:

In [28]:
# Use num_workers=0 to avoid common multiprocessing issues on Windows.
train_loader      = DataLoader(train_dataset,      batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
train_eval_loader = DataLoader(train_eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
val_loader        = DataLoader(val_dataset,        batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader       = DataLoader(test_dataset,       batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 112
Val batches: 16
Test batches: 32


In [29]:
images, labels = next(iter(train_loader))
print("Batch of images shape:", images.shape)
print("Batch of labels shape:", labels.shape)

Batch of images shape: torch.Size([32, 3, 224, 224])
Batch of labels shape: torch.Size([32])


### Class weights

Class weights help reduce the effect of class imbalance by assigning a larger loss weight to classes with fewer training images.

In [30]:
# Count training images in each class (same order as class_names: female = 0, male = 1, unknown = 2).
counts = np.bincount(train_dataset.targets, minlength=num_classes)
weights = len(train_dataset) / (num_classes * counts)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

for name, c, w in zip(class_names, counts, weights):
    print(f"{name:10s} images = {c:5d}   weight = {w:.2f}")

female     images =  1001   weight = 1.19
male       images =   724   weight = 1.64
unknown    images =  1836   weight = 0.65


# Model Building

Models are created on the CPU and moved to the GPU only for training to reduce GPU memory usage.

### Model -1 - CNN Model:

In [31]:
class CNN(nn.Module):
    def __init__(self, num_classes):
        super(CNN, self).__init__()

        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))

        self.block1 = block(3, 32)
        self.block2 = block(32, 64)
        self.block3 = block(64, 128)
        self.block4 = block(128, 256)
        self.block5 = block(256, 512)

        self.dropout_conv = nn.Dropout(0.1)   # CHANGED (was 0.3)
        self.gap = nn.AdaptiveAvgPool2d(1)       # Reduce each feature map to a 1x1 value.

        self.fc1 = nn.Linear(512, 256)
        self.dropout_fc = nn.Dropout(0.3)     # CHANGED (was 0.5)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.dropout_conv(x)

        x = self.gap(x)
        x = x.view(x.size(0), -1)                # Flatten features to [batch_size, 512].

        x = torch.relu(self.fc1(x))
        x = self.dropout_fc(x)
        x = self.fc2(x)                          # Return logits; CrossEntropyLoss applies log-softmax internally.
        return x

cnn_model = CNN(num_classes)
print(cnn_model)

CNN(
  (block1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block4): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), 

### Model - 2 - Transfer Learning (ResNet 50):

In [32]:
res_weights_name = "IMAGENET1K_V1" if PRETRAINED else None

resnet_model = models.resnet50(weights= res_weights_name)

# Freeze all layers before selectively unfreezing later layers.
for param in resnet_model.parameters():
    param.requires_grad = False

# Replace the final classifier with trainable layers.
resnet_model.fc = nn.Sequential(
                    nn.Linear(resnet_model.fc.in_features, 256),
                    nn.ReLU(),
                    nn.Dropout(0.5),
                    nn.Linear(256, num_classes)
)

# CHANGED: unfreeze layer3 and layer4 for fine-tuning (was layer4 only).
for layer in [resnet_model.layer3, resnet_model.layer4]:
    for param in layer.parameters():
        param.requires_grad = True

print(resnet_model.fc)

Sequential(
  (0): Linear(in_features=2048, out_features=256, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=256, out_features=3, bias=True)
)


### Model - 3 - Transfer Learning (VGG16):

In [33]:
vgg_weights_name = models.VGG16_Weights.DEFAULT

vgg_model = models.vgg16(weights= vgg_weights_name)

# Freeze all layers before selectively unfreezing later layers.
for param in vgg_model.parameters():
    param.requires_grad = False

# Replace the classifier; the flattened feature size is 512 * 7 * 7 = 25088.
vgg_model.classifier = nn.Sequential(
                    nn.Linear(25088, 256),
                    nn.ReLU(),
                    nn.Dropout(0.6),   # CHANGED (was 0.5)
                    nn.Linear(256, num_classes)
)

# Unfreeze the final VGG convolution block for fine-tuning.
for param in vgg_model.features[24:].parameters():                          # type: ignore
    param.requires_grad = True

print(vgg_model.classifier)

Sequential(
  (0): Linear(in_features=25088, out_features=256, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.6, inplace=False)
  (3): Linear(in_features=256, out_features=3, bias=True)
)


### Model - 4 - Transfer Learning (EfficientNet-B0):

In [34]:
weights_name = models.EfficientNet_B0_Weights.DEFAULT

effnet_model = models.efficientnet_b0(weights=weights_name)

# Freeze all layers before selectively unfreezing later layers.
for param in effnet_model.parameters():
    param.requires_grad = False

# Replace the classifier with a new classification head.
effnet_model.classifier = nn.Sequential(
                    nn.Dropout(0.4),   # CHANGED (was 0.3)
                    nn.Linear(effnet_model.classifier[1].in_features, 256), #type: ignore
                    nn.ReLU(),
                    nn.Dropout(0.6),   # CHANGED (was 0.5)
                    nn.Linear(256, num_classes)
)

# Unfreeze the final EfficientNet feature blocks for fine-tuning.
for param in effnet_model.features[7:].parameters():
    param.requires_grad = True

print(effnet_model.classifier)

Sequential(
  (0): Dropout(p=0.4, inplace=False)
  (1): Linear(in_features=1280, out_features=256, bias=True)
  (2): ReLU()
  (3): Dropout(p=0.6, inplace=False)
  (4): Linear(in_features=256, out_features=3, bias=True)
)


### Model - 5 - Vision Transformer (ViT):

In [35]:
vit_model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT if PRETRAINED else None)

# Freeze all layers before selectively unfreezing later layers.
for param in vit_model.parameters():
    param.requires_grad = False

# Replace the classification head.
vit_model.heads = nn.Sequential(
    nn.Linear(vit_model.heads[0].in_features, 256),                     # type: ignore
    nn.ReLU(),
    nn.Dropout(0.6),   # CHANGED (was 0.5)
    nn.Linear(256, num_classes)
)

# Unfreeze the final transformer encoder block for fine-tuning.
for param in vit_model.encoder.layers[-1].parameters():
    param.requires_grad = True
for param in vit_model.encoder.ln.parameters():
    param.requires_grad = True

print(vit_model.heads)

Sequential(
  (0): Linear(in_features=768, out_features=256, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.6, inplace=False)
  (3): Linear(in_features=256, out_features=3, bias=True)
)


# Model Training:

### Training function

- Each epoch performs training followed by validation.
- The learning rate is reduced when validation loss stops improving.
- Training stops early if validation loss does not improve for five consecutive epochs, and the best weights are restored.

In [36]:
def train_model(model, optimizer, criterion, num_epochs, patience=7):      # CHANGED: patience is now an argument (default 7)

    history = {
        "train_acc": [],
        "val_acc": [],
        "train_loss": [],
        "val_loss": [],
        "val_f1": []
    }

    best_f1 = -1.0                                                          # CHANGED: select best epoch by validation macro-F1
    best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    best_epoch = 0

    epochs_without_improvement = 0

    # Reduce learning rate if validation macro-F1 stops improving
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",                                                         # CHANGED (was "min")
        factor=0.5,
        patience=2
    )

    for epoch in range(num_epochs):

        # Training
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

        train_acc = correct / total
        train_loss = running_loss / total
        history["train_acc"].append(train_acc)
        history["train_loss"].append(train_loss)

        # Validation
        model.eval()
        val_running_loss, val_correct, val_total = 0.0, 0, 0
        all_p, all_l = [], []                                               # CHANGED

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_running_loss += loss.item() * images.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += labels.size(0)
                all_p.extend(outputs.argmax(1).cpu().numpy())               # CHANGED
                all_l.extend(labels.cpu().numpy())                          # CHANGED

        val_loss = val_running_loss / val_total
        val_acc = val_correct / val_total
        val_f1 = f1_score(all_l, all_p, average="macro", zero_division=0)   # CHANGED
        history["val_acc"].append(val_acc)
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_f1)

        scheduler.step(val_f1)                                              # CHANGED

        print(
            f"Epoch {epoch + 1}/{num_epochs}: "
            f"Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, "
            f"Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}, Val F1={val_f1:.4f}"
        )

        # Keep the best weights (by validation macro-F1) and apply early stopping.
        if val_f1 > best_f1:                                                # CHANGED
            best_f1 = val_f1
            best_epoch = epoch + 1
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

    print(f"Best epoch: {best_epoch} (val macro-F1 = {best_f1:.4f})")
    model.load_state_dict(best_weights)
    return history

### CNN Model:

In [37]:
cnn_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(cnn_model.parameters(), lr=0.001, weight_decay=1e-4)

cnn_history = train_model(cnn_model, optimizer, criterion, num_epochs=50)

cnn_model.to("cpu")
torch.cuda.empty_cache()

Epoch 1/50: 100%|██████████| 112/112 [01:34<00:00,  1.19it/s]


Epoch 1/50: Train Loss=0.6627, Val Loss=0.6663, Train Acc=0.7096, Val Acc=0.7421, Val F1=0.6245


Epoch 2/50: 100%|██████████| 112/112 [01:16<00:00,  1.47it/s]


Epoch 2/50: Train Loss=0.5843, Val Loss=0.3735, Train Acc=0.7284, Val Acc=0.7815, Val F1=0.6533


Epoch 3/50: 100%|██████████| 112/112 [01:19<00:00,  1.41it/s]


Epoch 3/50: Train Loss=0.5875, Val Loss=0.4377, Train Acc=0.7256, Val Acc=0.7500, Val F1=0.6714


... (54 outputs left out)

Epoch 31/50: 100%|██████████| 112/112 [01:21<00:00,  1.38it/s]


Epoch 31/50: Train Loss=0.4813, Val Loss=0.3621, Train Acc=0.7888, Val Acc=0.7972, Val F1=0.7338


Epoch 32/50: 100%|██████████| 112/112 [01:14<00:00,  1.49it/s]


Epoch 32/50: Train Loss=0.4771, Val Loss=0.3534, Train Acc=0.7958, Val Acc=0.8012, Val F1=0.7343
Early stopping at epoch 32
Best epoch: 25 (val macro-F1 = 0.7480)


In [38]:
# Saving the Cnn Model:
torch.save( cnn_model.state_dict(),
            r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\gender\custom_cnn_model_gender.pth"
            )
print("Custom CNN model Gender saved successfully!")

Custom CNN model Gender saved successfully!


### ResNet 50:

In [39]:
resnet_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW([
                        {"params": resnet_model.layer4.parameters(), "lr": 1e-5},
                        {"params": resnet_model.fc.parameters(), "lr": 1e-3}
                        ], weight_decay=0.01)

resnet_history = train_model(resnet_model, optimizer, criterion, num_epochs=30)

resnet_model.to("cpu")
torch.cuda.empty_cache()

Epoch 1/30: 100%|██████████| 112/112 [01:35<00:00,  1.18it/s]


Epoch 1/30: Train Loss=0.4163, Val Loss=0.1813, Train Acc=0.8557, Val Acc=0.9331, Val F1=0.9115


Epoch 2/30: 100%|██████████| 112/112 [01:32<00:00,  1.21it/s]


Epoch 2/30: Train Loss=0.2858, Val Loss=0.1759, Train Acc=0.9048, Val Acc=0.9272, Val F1=0.9029


Epoch 3/30: 100%|██████████| 112/112 [01:33<00:00,  1.19it/s]


Epoch 3/30: Train Loss=0.2367, Val Loss=0.1901, Train Acc=0.9250, Val Acc=0.9193, Val F1=0.8925


... (6 outputs left out)

Epoch 7/30: 100%|██████████| 112/112 [01:31<00:00,  1.22it/s]


Epoch 7/30: Train Loss=0.1389, Val Loss=0.2538, Train Acc=0.9565, Val Acc=0.9173, Val F1=0.8828


Epoch 8/30: 100%|██████████| 112/112 [01:33<00:00,  1.20it/s]


Epoch 8/30: Train Loss=0.1119, Val Loss=0.2337, Train Acc=0.9680, Val Acc=0.9173, Val F1=0.8877
Early stopping at epoch 8
Best epoch: 1 (val macro-F1 = 0.9115)


In [40]:
# Saving the ResNet 50 Model:
torch.save( resnet_model.state_dict(),
            r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\gender\resnet50_model_gender.pth"
            )
print("Transfer Learning ResNet50 Model Gender saved successfully!")

Transfer Learning ResNet50 Model Gender saved successfully!


### VGG16:

In [ ]:
vgg_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)   # CHANGED: label smoothing

optimizer = optim.AdamW([
                        {"params": vgg_model.features[24:].parameters(), "lr": 1e-5},   #type: ignore
                        {"params": vgg_model.classifier.parameters(), "lr": 1e-3}
                        ], weight_decay=0.01)

vgg_history = train_model(vgg_model, optimizer, criterion, num_epochs=25)

vgg_model.to("cpu")
torch.cuda.empty_cache()

Epoch 1/25: 100%|██████████| 112/112 [01:44<00:00,  1.07it/s]


Epoch 1/25: Train Loss=0.7388, Val Loss=0.5467, Train Acc=0.8405, Val Acc=0.9154, Val F1=0.8887


Epoch 2/25: 100%|██████████| 112/112 [01:41<00:00,  1.10it/s]


Epoch 2/25: Train Loss=0.5457, Val Loss=0.5547, Train Acc=0.9174, Val Acc=0.9134, Val F1=0.8788


Epoch 3/25: 100%|██████████| 112/112 [01:38<00:00,  1.13it/s]


Epoch 3/25: Train Loss=0.4899, Val Loss=0.5464, Train Acc=0.9379, Val Acc=0.9134, Val F1=0.8819


... (24 outputs left out)

Epoch 16/25: 100%|██████████| 112/112 [01:37<00:00,  1.15it/s]


Epoch 16/25: Train Loss=0.3834, Val Loss=0.5272, Train Acc=0.9896, Val Acc=0.9134, Val F1=0.8829


Epoch 17/25: 100%|██████████| 112/112 [01:39<00:00,  1.12it/s]


Epoch 17/25: Train Loss=0.3796, Val Loss=0.5276, Train Acc=0.9944, Val Acc=0.9173, Val F1=0.8880
Early stopping at epoch 17
Best epoch: 10 (val macro-F1 = 0.8988)


In [42]:
# Saving the VGG16 Model:
torch.save( vgg_model.state_dict(),
            r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\gender\vgg16_model_gender.pth"
            )
print("Transfer Learning VGG16 Model Gender saved successfully!")

Transfer Learning VGG16 Model Gender saved successfully!


### EfficientNet-B0:

In [43]:
effnet_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)   # CHANGED: label smoothing

optimizer = optim.AdamW([
                        {"params": effnet_model.features[7:].parameters(), "lr": 1e-5},
                        {"params": effnet_model.classifier.parameters(), "lr": 1e-3}
                        ], weight_decay=0.01)

effnet_history = train_model(effnet_model, optimizer, criterion, num_epochs=10)

effnet_model.to("cpu")
torch.cuda.empty_cache()

Epoch 1/10: 100%|██████████| 112/112 [01:15<00:00,  1.48it/s]


Epoch 1/10: Train Loss=0.6799, Val Loss=0.5713, Train Acc=0.8340, Val Acc=0.8839, Val F1=0.8414


Epoch 2/10: 100%|██████████| 112/112 [01:13<00:00,  1.52it/s]


Epoch 2/10: Train Loss=0.6056, Val Loss=0.5714, Train Acc=0.8666, Val Acc=0.8957, Val F1=0.8555


Epoch 3/10: 100%|██████████| 112/112 [01:13<00:00,  1.52it/s]


Epoch 3/10: Train Loss=0.6017, Val Loss=0.5561, Train Acc=0.8801, Val Acc=0.8996, Val F1=0.8663


... (10 outputs left out)

Epoch 9/10: 100%|██████████| 112/112 [01:12<00:00,  1.54it/s]


Epoch 9/10: Train Loss=0.5520, Val Loss=0.5450, Train Acc=0.9020, Val Acc=0.8957, Val F1=0.8629


Epoch 10/10: 100%|██████████| 112/112 [01:12<00:00,  1.54it/s]


Epoch 10/10: Train Loss=0.5519, Val Loss=0.5395, Train Acc=0.9023, Val Acc=0.9094, Val F1=0.8788
Best epoch: 7 (val macro-F1 = 0.8811)


In [44]:
# Saving the EfficientNet-B0 Model:
torch.save( effnet_model.state_dict(),
            r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\gender\efficientnet_b0_model_gender.pth"
            )
print("Transfer Learning EfficientNet-B0 Model Gender saved successfully!")

Transfer Learning EfficientNet-B0 Model Gender saved successfully!


### ViT:

In [45]:
vit_model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)   # CHANGED: label smoothing

optimizer = optim.AdamW([
                        {"params": vit_model.encoder.layers[-1].parameters(), "lr": 5e-6},   # CHANGED (was 1e-5)
                        {"params": vit_model.encoder.ln.parameters(), "lr": 5e-6},
                        {"params": vit_model.heads.parameters(), "lr": 1e-3}
                        ], weight_decay=0.1)   # CHANGED (was 0.05)

vit_history = train_model(vit_model, optimizer, criterion, num_epochs=25)   # CHANGED (was 10)

vit_model.to("cpu")
torch.cuda.empty_cache()

Epoch 1/25: 100%|██████████| 112/112 [02:50<00:00,  1.52s/it]


Epoch 1/25: Train Loss=0.5441, Val Loss=0.5236, Train Acc=0.9045, Val Acc=0.9134, Val F1=0.8884


Epoch 2/25: 100%|██████████| 112/112 [02:50<00:00,  1.52s/it]


Epoch 2/25: Train Loss=0.4726, Val Loss=0.5149, Train Acc=0.9413, Val Acc=0.9252, Val F1=0.9002


Epoch 3/25: 100%|██████████| 112/112 [02:48<00:00,  1.51s/it]


Epoch 3/25: Train Loss=0.4526, Val Loss=0.5153, Train Acc=0.9469, Val Acc=0.9311, Val F1=0.9089


... (22 outputs left out)

Epoch 15/25: 100%|██████████| 112/112 [02:51<00:00,  1.53s/it]


Epoch 15/25: Train Loss=0.3782, Val Loss=0.5043, Train Acc=0.9876, Val Acc=0.9252, Val F1=0.9014


Epoch 16/25: 100%|██████████| 112/112 [02:51<00:00,  1.53s/it]


Epoch 16/25: Train Loss=0.3746, Val Loss=0.5046, Train Acc=0.9907, Val Acc=0.9291, Val F1=0.9059
Early stopping at epoch 16
Best epoch: 9 (val macro-F1 = 0.9125)


In [46]:
# Saving the ViT Model:
torch.save( vit_model.state_dict(),
            r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\gender\vit_model_gender.pth"
            )
print("ViT Model Gender saved successfully!")

ViT Model Gender saved successfully!


### Training curves (overfitting check):

In [47]:
histories = {}
for name, var in [("CNN", "cnn"), ("ResNet50", "resnet"), ("VGG16", "vgg"), ("EfficientNet_B0", "effnet"), ("ViT_B16", "vit")]:
    if var + "_history" in globals():
        histories[name] = globals()[var + "_history"]

fig = go.Figure()
for name, h in histories.items():
    fig.add_scatter(y=h["train_loss"], mode="lines", name=name + " train")
    fig.add_scatter(y=h["val_loss"], mode="lines", line=dict(dash="dash"), name=name + " val")
fig.update_layout(height=450, width=850, title="Loss per Epoch (solid = train, dashed = validation)",
                  xaxis_title="Epoch", yaxis_title="Loss")
fig.show()

fig = go.Figure()
for name, h in histories.items():
    fig.add_scatter(y=h["val_acc"], mode="lines", name=name)
fig.update_layout(height=450, width=850, title="Validation Accuracy per Epoch",
                  xaxis_title="Epoch", yaxis_title="Accuracy")
fig.show()

# Model Evaluation:

In [48]:
def evaluate_model(model, loader, dataset_name="Set", show_report=False):
    model.to(device)
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images.to(device))
            probs = torch.softmax(outputs, dim=1)

            all_labels.extend(labels.numpy())
            all_preds.extend(probs.argmax(1).cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    model.to("cpu")
    torch.cuda.empty_cache()

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    rec = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    if num_classes == 2:
        auc = roc_auc_score(all_labels, all_probs[:, 1])
    else:
        auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")

    print(f"--- {dataset_name} Metrics ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC-ROC:   {auc:.4f}\n")

    if show_report:
        print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "auc": auc,
            "labels": all_labels, "preds": all_preds, "probs": all_probs}

### CNN Model Result:

In [49]:
print("===== CNN RESULTS =====\n")
cnn_train_results = evaluate_model(cnn_model, train_eval_loader, "Train")
cnn_val_results = evaluate_model(cnn_model, val_loader, "Validation")
cnn_test_results = evaluate_model(cnn_model, test_loader, "Test", show_report=True)

===== CNN RESULTS =====

--- Train Metrics ---
Accuracy:  0.8076
Precision: 0.7417
Recall:    0.7468
F1-Score:  0.7435
AUC-ROC:   0.9273

--- Validation Metrics ---
Accuracy:  0.8130
Precision: 0.7470
Recall:    0.7496
F1-Score:  0.7480
AUC-ROC:   0.9262

--- Test Metrics ---
Accuracy:  0.7782
Precision: 0.7059
Recall:    0.7109
F1-Score:  0.7070
AUC-ROC:   0.9078

              precision    recall  f1-score   support

      female       0.64      0.62      0.63       300
        male       0.48      0.56      0.52       198
     unknown       1.00      0.95      0.98       521

    accuracy                           0.78      1019
   macro avg       0.71      0.71      0.71      1019
weighted avg       0.79      0.78      0.78      1019



### ResNet 50 Model Result:

In [50]:
print("===== RESNET50 RESULTS =====\n")
resnet_train_results = evaluate_model(resnet_model, train_eval_loader, "Train")
resnet_val_results = evaluate_model(resnet_model, val_loader, "Validation")
resnet_test_results = evaluate_model(resnet_model, test_loader, "Test", show_report=True)

===== RESNET50 RESULTS =====

--- Train Metrics ---
Accuracy:  0.9312
Precision: 0.9061
Recall:    0.9194
F1-Score:  0.9122
AUC-ROC:   0.9883

--- Validation Metrics ---
Accuracy:  0.9331
Precision: 0.9098
Recall:    0.9133
F1-Score:  0.9115
AUC-ROC:   0.9855

--- Test Metrics ---
Accuracy:  0.9146
Precision: 0.8847
Recall:    0.8927
F1-Score:  0.8884
AUC-ROC:   0.9832

              precision    recall  f1-score   support

      female       0.86      0.89      0.88       300
        male       0.80      0.82      0.81       198
     unknown       1.00      0.96      0.98       521

    accuracy                           0.91      1019
   macro avg       0.88      0.89      0.89      1019
weighted avg       0.92      0.91      0.92      1019



### VGG16 Model Result:

In [51]:
print("===== VGG16 RESULTS =====\n")
vgg_train_results = evaluate_model(vgg_model, train_eval_loader, "Train")
vgg_val_results = evaluate_model(vgg_model, val_loader, "Validation")
vgg_test_results = evaluate_model(vgg_model, test_loader, "Test", show_report=True)

===== VGG16 RESULTS =====

--- Train Metrics ---
Accuracy:  0.9882
Precision: 0.9832
Recall:    0.9876
F1-Score:  0.9854
AUC-ROC:   0.9993

--- Validation Metrics ---
Accuracy:  0.9252
Precision: 0.9053
Recall:    0.8947
F1-Score:  0.8988
AUC-ROC:   0.9747

--- Test Metrics ---
Accuracy:  0.9176
Precision: 0.8915
Recall:    0.8935
F1-Score:  0.8923
AUC-ROC:   0.9719

              precision    recall  f1-score   support

      female       0.86      0.90      0.88       300
        male       0.83      0.81      0.82       198
     unknown       0.99      0.97      0.98       521

    accuracy                           0.92      1019
   macro avg       0.89      0.89      0.89      1019
weighted avg       0.92      0.92      0.92      1019



### EfficientNet-B0 Model Result:

In [52]:
print("===== EFFICIENTNET_B0 RESULTS =====\n")
effnet_train_results = evaluate_model(effnet_model, train_eval_loader, "Train")
effnet_val_results = evaluate_model(effnet_model, val_loader, "Validation")
effnet_test_results = evaluate_model(effnet_model, test_loader, "Test", show_report=True)

===== EFFICIENTNET_B0 RESULTS =====

--- Train Metrics ---
Accuracy:  0.9351
Precision: 0.9121
Recall:    0.9190
F1-Score:  0.9154
AUC-ROC:   0.9900

--- Validation Metrics ---
Accuracy:  0.9114
Precision: 0.8830
Recall:    0.8794
F1-Score:  0.8811
AUC-ROC:   0.9747

--- Test Metrics ---
Accuracy:  0.8999
Precision: 0.8675
Recall:    0.8697
F1-Score:  0.8682
AUC-ROC:   0.9759

              precision    recall  f1-score   support

      female       0.83      0.89      0.86       300
        male       0.78      0.76      0.77       198
     unknown       0.99      0.96      0.97       521

    accuracy                           0.90      1019
   macro avg       0.87      0.87      0.87      1019
weighted avg       0.90      0.90      0.90      1019



### ViT Model Result:

In [53]:
print("===== VIT_B16 RESULTS =====\n")
vit_train_results = evaluate_model(vit_model, train_eval_loader, "Train")
vit_val_results = evaluate_model(vit_model, val_loader, "Validation")
vit_test_results = evaluate_model(vit_model, test_loader, "Test", show_report=True)

===== VIT_B16 RESULTS =====

--- Train Metrics ---
Accuracy:  0.9896
Precision: 0.9854
Recall:    0.9884
F1-Score:  0.9869
AUC-ROC:   0.9996

--- Validation Metrics ---
Accuracy:  0.9331
Precision: 0.9151
Recall:    0.9102
F1-Score:  0.9125
AUC-ROC:   0.9807

--- Test Metrics ---
Accuracy:  0.9333
Precision: 0.9145
Recall:    0.9165
F1-Score:  0.9154
AUC-ROC:   0.9785

              precision    recall  f1-score   support

      female       0.89      0.92      0.90       300
        male       0.87      0.86      0.87       198
     unknown       0.98      0.97      0.98       521

    accuracy                           0.93      1019
   macro avg       0.91      0.92      0.92      1019
weighted avg       0.93      0.93      0.93      1019



### Result Summary:

In [54]:
def create_results_summary(train_results, val_results, test_results):

    return pd.DataFrame({
        "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "AUC-ROC"],
        "Train": [train_results["accuracy"], train_results["precision"], train_results["recall"],
                  train_results["f1"], train_results["auc"]],
        "Validation": [val_results["accuracy"], val_results["precision"], val_results["recall"],
                       val_results["f1"], val_results["auc"]],
        "Test": [test_results["accuracy"], test_results["precision"], test_results["recall"],
                 test_results["f1"], test_results["auc"]]
    })

In [55]:
model_list = [("CNN", "cnn"), ("ResNet50", "resnet"), ("VGG16", "vgg"), ("EfficientNet_B0", "effnet"), ("ViT_B16", "vit")]

for name, var in model_list:
    if var + "_test_results" in globals():
        print(f"\n===== {name.upper()} RESULTS =====\n")
        print(create_results_summary(globals()[var + "_train_results"],
                                     globals()[var + "_val_results"],
                                     globals()[var + "_test_results"]))


===== CNN RESULTS =====

      Metric     Train  Validation      Test
0   Accuracy  0.807638    0.812992  0.778214
1  Precision  0.741682    0.747021  0.705875
2     Recall  0.746847    0.749587  0.710874
3   F1-Score  0.743510    0.748040  0.707046
4    AUC-ROC  0.927348    0.926240  0.907797

===== RESNET50 RESULTS =====

      Metric     Train  Validation      Test
0   Accuracy  0.931199    0.933071  0.914622
1  Precision  0.906090    0.909825  0.884673
2     Recall  0.919422    0.913257  0.892726
3   F1-Score  0.912212    0.911485  0.888426
4    AUC-ROC  0.988290    0.985479  0.983167

===== VGG16 RESULTS =====

      Metric     Train  Validation      Test
0   Accuracy  0.988206    0.925197  0.917566
1  Precision  0.983196    0.905337  0.891521
2     Recall  0.987617    0.894741  0.893501
3   F1-Score  0.985366    0.898769  0.892332
4    AUC-ROC  0.999309    0.974708  0.971856

===== EFFICIENTNET_B0 RESULTS =====

      Metric     Train  Validation      Test
0   Accuracy  0.935131

# Confusion Matrix:

The class order is `female = 0`, `male = 1`, `unknown = 2` so the axis labels are `['FEMALE', 'MALE', 'UNKNOWN']`.


### CNN Model:

In [ ]:
# Train Confusion Matrix
cm_cnn_train = confusion_matrix(
                        cnn_train_results["labels"],
                        cnn_train_results["preds"]
                    )

cnn_train_fig = px.imshow(
                        cm_cnn_train,
                        text_auto=True,
                        x=['FEMALE', 'MALE', 'UNKNOWN'], 
                        y=['FEMALE', 'MALE', 'UNKNOWN'],
                        color_continuous_scale='Blues',
                        title='Confusion Matrix for CNN Model - Train'
                    )

cnn_train_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
cnn_train_fig.show()

joblib.dump(cnn_train_fig, CHARTS_DIR / "confusion_matrix_gender_cnn_train.pkl")


# Validation Confusion Matrix
cm_cnn_val = confusion_matrix(
                        cnn_val_results["labels"],
                        cnn_val_results["preds"]
                    )

cnn_val_fig = px.imshow(
                        cm_cnn_val,
                        text_auto=True,
                        x=['FEMALE', 'MALE', 'UNKNOWN'],      # female = 0, male = 1
                        y=['FEMALE', 'MALE', 'UNKNOWN'],
                        color_continuous_scale='Blues',
                        title='Confusion Matrix for CNN Model - Validation'
                    )

cnn_val_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
cnn_val_fig.show()

joblib.dump(cnn_val_fig, CHARTS_DIR / "confusion_matrix_gender_cnn_val.pkl")


# Test Confusion Matrix
cm_cnn_test = confusion_matrix(
                        cnn_test_results["labels"],
                        cnn_test_results["preds"]
                    )

cnn_test_fig = px.imshow(
                        cm_cnn_test,
                        text_auto=True,
                        x=['FEMALE', 'MALE', 'UNKNOWN'],      # female = 0, male = 1
                        y=['FEMALE', 'MALE', 'UNKNOWN'],
                        color_continuous_scale='Blues',
                        title='Confusion Matrix for CNN Model - Test'
                    )

cnn_test_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
cnn_test_fig.show()

joblib.dump(cnn_test_fig, CHARTS_DIR / "confusion_matrix_gender_cnn_test.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\gender\\confusion_matrix_gender_cnn_test.pkl']

### ResNet50 Model:

In [ ]:
# Train Confusion Matrix
cm_resnet_train = confusion_matrix(
                        resnet_train_results["labels"],
                        resnet_train_results["preds"]
                    )

resnet_train_fig = px.imshow(
                        cm_resnet_train,
                        text_auto=True,
                        x=['FEMALE', 'MALE', 'UNKNOWN'],
                        y=['FEMALE', 'MALE', 'UNKNOWN'],
                        color_continuous_scale='Greens',
                        title='Confusion Matrix for ResNet50 Model - Train'
                    )

resnet_train_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
resnet_train_fig.show()
joblib.dump(resnet_train_fig, CHARTS_DIR / "confusion_matrix_gender_resnet50_train.pkl")

# Validation Confusion Matrix
cm_resnet_val = confusion_matrix(
                        resnet_val_results["labels"],
                        resnet_val_results["preds"]
                    )

resnet_val_fig = px.imshow(
                        cm_resnet_val,
                        text_auto=True,
                        x=['FEMALE', 'MALE', 'UNKNOWN'], 
                        y=['FEMALE', 'MALE', 'UNKNOWN'],
                        color_continuous_scale='Greens',
                        title='Confusion Matrix for ResNet50 Model - Validation'
                    )

resnet_val_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
resnet_val_fig.show()
joblib.dump(resnet_val_fig, CHARTS_DIR / "confusion_matrix_gender_resnet50_val.pkl")

# Test Confusion Matrix
cm_resnet_test = confusion_matrix(
                        resnet_test_results["labels"],
                        resnet_test_results["preds"]
                    )

resnet_test_fig = px.imshow(
                        cm_resnet_test,
                        text_auto=True,
                        x=['FEMALE', 'MALE', 'UNKNOWN'], 
                        y=['FEMALE', 'MALE', 'UNKNOWN'],
                        color_continuous_scale='Greens',
                        title='Confusion Matrix for ResNet50 Model - Test'
                    )

resnet_test_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
resnet_test_fig.show()
joblib.dump(resnet_test_fig, CHARTS_DIR / "confusion_matrix_gender_resnet50_test.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\gender\\confusion_matrix_gender_resnet50_test.pkl']

### VGG16 Model:

In [ ]:
# Train Confusion Matrix
cm_vgg_train = confusion_matrix(
                        vgg_train_results["labels"],
                        vgg_train_results["preds"]
                    )

vgg_train_fig = px.imshow(
                        cm_vgg_train,
                        text_auto=True,
                        x=['FEMALE', 'MALE', 'UNKNOWN'],
                        y=['FEMALE', 'MALE', 'UNKNOWN'],
                        color_continuous_scale='Reds',
                        title='Confusion Matrix for VGG16 Model - Train'
                    )

vgg_train_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
vgg_train_fig.show()
joblib.dump(vgg_train_fig, CHARTS_DIR / "confusion_matrix_gender_vgg_16_train.pkl")

# Validation Confusion Matrix
cm_vgg_val = confusion_matrix(
                        vgg_val_results["labels"],
                        vgg_val_results["preds"]
                    )

vgg_val_fig = px.imshow(
                        cm_vgg_val,
                        text_auto=True,
                        x=['FEMALE', 'MALE', 'UNKNOWN'],
                        y=['FEMALE', 'MALE', 'UNKNOWN'],
                        color_continuous_scale='Reds',
                        title='Confusion Matrix for VGG16 Model - Validation'
                    )

vgg_val_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
vgg_val_fig.show()
joblib.dump(vgg_val_fig, CHARTS_DIR / "confusion_matrix_gender_vgg_16_val.pkl")

# Test Confusion Matrix
cm_vgg_test = confusion_matrix(
                        vgg_test_results["labels"],
                        vgg_test_results["preds"]
                    )

vgg_test_fig = px.imshow(
                        cm_vgg_test,
                        text_auto=True,
                        x=['FEMALE', 'MALE', 'UNKNOWN'],
                        y=['FEMALE', 'MALE', 'UNKNOWN'],
                        color_continuous_scale='Reds',
                        title='Confusion Matrix for VGG16 Model - Test'
                    )

vgg_test_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
vgg_test_fig.show()
joblib.dump(vgg_test_fig, CHARTS_DIR / "confusion_matrix_gender_vgg_16_test.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\gender\\confusion_matrix_gender_vgg_16_test.pkl']

### EfficientNet_B0 Model:

In [ ]:
# Train Confusion Matrix
cm_effnet_train = confusion_matrix(
                        effnet_train_results["labels"],
                        effnet_train_results["preds"]
                    )

effnet_train_fig = px.imshow(
                        cm_effnet_train,
                        text_auto=True,
                        x=['FEMALE', 'MALE', 'UNKNOWN'], 
                        y=['FEMALE', 'MALE', 'UNKNOWN'],
                        color_continuous_scale='Purples',
                        title='Confusion Matrix for EfficientNet_B0 Model - Train'
                    )

effnet_train_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
effnet_train_fig.show()
joblib.dump(effnet_train_fig, CHARTS_DIR / "confusion_matrix_gender_EffNet_b0_train.pkl")

# Validation Confusion Matrix
cm_effnet_val = confusion_matrix(
                        effnet_val_results["labels"],
                        effnet_val_results["preds"]
                    )

effnet_val_fig = px.imshow(
                        cm_effnet_val,
                        text_auto=True,
                        x=['FEMALE', 'MALE', 'UNKNOWN'], 
                        y=['FEMALE', 'MALE', 'UNKNOWN'],
                        color_continuous_scale='Purples',
                        title='Confusion Matrix for EfficientNet_B0 Model - Validation'
                    )

effnet_val_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
effnet_val_fig.show()
joblib.dump(effnet_val_fig, CHARTS_DIR / "confusion_matrix_gender_EffNet_b0_val.pkl")

# Test Confusion Matrix
cm_effnet_test = confusion_matrix(
                        effnet_test_results["labels"],
                        effnet_test_results["preds"]
                    )

effnet_test_fig = px.imshow(
                        cm_effnet_test,
                        text_auto=True,
                        x=['FEMALE', 'MALE', 'UNKNOWN'],
                        y=['FEMALE', 'MALE', 'UNKNOWN'],
                        color_continuous_scale='Purples',
                        title='Confusion Matrix for EfficientNet_B0 Model - Test'
                    )

effnet_test_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
effnet_test_fig.show()
joblib.dump(effnet_test_fig, CHARTS_DIR / "confusion_matrix_gender_EffNet_b0_test.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\gender\\confusion_matrix_gender_EffNet_b0_test.pkl']

### ViT_B16 Model:

In [ ]:
# Train Confusion Matrix
cm_vit_train = confusion_matrix(
                        vit_train_results["labels"],
                        vit_train_results["preds"]
                    )

vit_train_fig = px.imshow(
                        cm_vit_train,
                        text_auto=True,
                        x=['FEMALE', 'MALE', 'UNKNOWN'], 
                        y=['FEMALE', 'MALE', 'UNKNOWN'],
                        color_continuous_scale='Oranges',
                        title='Confusion Matrix for ViT_B16 Model - Train'
                    )

vit_train_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
vit_train_fig.show()
joblib.dump(vit_train_fig, CHARTS_DIR / "confusion_matrix_gender_ViT_B16_train.pkl")

# Validation Confusion Matrix
cm_vit_val = confusion_matrix(
                        vit_val_results["labels"],
                        vit_val_results["preds"]
                    )

vit_val_fig = px.imshow(
                        cm_vit_val,
                        text_auto=True,
                        x=['FEMALE', 'MALE', 'UNKNOWN'],
                        y=['FEMALE', 'MALE', 'UNKNOWN'],
                        color_continuous_scale='Oranges',
                        title='Confusion Matrix for ViT_B16 Model - Validation'
                    )

vit_val_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
vit_val_fig.show()
joblib.dump(vit_val_fig, CHARTS_DIR / "confusion_matrix_gender_ViT_B16_val.pkl")

# Test Confusion Matrix
cm_vit_test = confusion_matrix(
                        vit_test_results["labels"],
                        vit_test_results["preds"]
                    )

vit_test_fig = px.imshow(
                        cm_vit_test,
                        text_auto=True,
                        x=['FEMALE', 'MALE', 'UNKNOWN'],  
                        y=['FEMALE', 'MALE', 'UNKNOWN'],
                        color_continuous_scale='Oranges',
                        title='Confusion Matrix for ViT_B16 Model - Test'
                    )

vit_test_fig.update_layout(
                        height=500, width=500,
                        xaxis_title='Predicted',
                        yaxis_title='Actual'
                    )
vit_test_fig.show()
joblib.dump(vit_test_fig, CHARTS_DIR / "confusion_matrix_gender_ViT_B16_test.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\gender\\confusion_matrix_gender_ViT_B16_test.pkl']

# ROC Curve

Train, validation, and test ROC curves with the AUC score of each. The curves use the predicted probability of class 1 (`male`).


### CNN Model:

In [70]:
# Train - Female
fpr_tr_female, tpr_tr_female, _ = roc_curve(
    (cnn_train_results["labels"] == 0).astype(int),
    cnn_train_results["probs"][:, 0]
)

# Train - Male
fpr_tr_male, tpr_tr_male, _ = roc_curve(
    (cnn_train_results["labels"] == 1).astype(int),
    cnn_train_results["probs"][:, 1]
)

# Train - Unknown
fpr_tr_unknown, tpr_tr_unknown, _ = roc_curve(
    (cnn_train_results["labels"] == 2).astype(int),
    cnn_train_results["probs"][:, 2]
)

# Validation - Female
fpr_va_female, tpr_va_female, _ = roc_curve(
    (cnn_val_results["labels"] == 0).astype(int),
    cnn_val_results["probs"][:, 0]
)

# Validation - Male
fpr_va_male, tpr_va_male, _ = roc_curve(
    (cnn_val_results["labels"] == 1).astype(int),
    cnn_val_results["probs"][:, 1]
)

# Validation - Unknown
fpr_va_unknown, tpr_va_unknown, _ = roc_curve(
    (cnn_val_results["labels"] == 2).astype(int),
    cnn_val_results["probs"][:, 2]
)

# Test - Female
fpr_te_female, tpr_te_female, _ = roc_curve(
    (cnn_test_results["labels"] == 0).astype(int),
    cnn_test_results["probs"][:, 0]
)

# Test - Male
fpr_te_male, tpr_te_male, _ = roc_curve(
    (cnn_test_results["labels"] == 1).astype(int),
    cnn_test_results["probs"][:, 1]
)

# Test - Unknown
fpr_te_unknown, tpr_te_unknown, _ = roc_curve(
    (cnn_test_results["labels"] == 2).astype(int),
    cnn_test_results["probs"][:, 2]
)

fig_cnn_roc = go.Figure()

fig_cnn_roc.add_scatter(x=fpr_tr_female, y=tpr_tr_female, mode="lines",
    name=f"Train Female (AUC={auc(fpr_tr_female, tpr_tr_female):.3f})")

fig_cnn_roc.add_scatter(x=fpr_tr_male, y=tpr_tr_male, mode="lines",
    name=f"Train Male (AUC={auc(fpr_tr_male, tpr_tr_male):.3f})")

fig_cnn_roc.add_scatter(x=fpr_tr_unknown, y=tpr_tr_unknown, mode="lines",
    name=f"Train Unknown (AUC={auc(fpr_tr_unknown, tpr_tr_unknown):.3f})")

fig_cnn_roc.add_scatter(x=fpr_va_female, y=tpr_va_female, mode="lines",
    name=f"Validation Female (AUC={auc(fpr_va_female, tpr_va_female):.3f})")

fig_cnn_roc.add_scatter(x=fpr_va_male, y=tpr_va_male, mode="lines",
    name=f"Validation Male (AUC={auc(fpr_va_male, tpr_va_male):.3f})")

fig_cnn_roc.add_scatter(x=fpr_va_unknown, y=tpr_va_unknown, mode="lines",
    name=f"Validation Unknown (AUC={auc(fpr_va_unknown, tpr_va_unknown):.3f})")

fig_cnn_roc.add_scatter(x=fpr_te_female, y=tpr_te_female, mode="lines",
    name=f"Test Female (AUC={auc(fpr_te_female, tpr_te_female):.3f})")

fig_cnn_roc.add_scatter(x=fpr_te_male, y=tpr_te_male, mode="lines",
    name=f"Test Male (AUC={auc(fpr_te_male, tpr_te_male):.3f})")

fig_cnn_roc.add_scatter(x=fpr_te_unknown, y=tpr_te_unknown, mode="lines",
    name=f"Test Unknown (AUC={auc(fpr_te_unknown, tpr_te_unknown):.3f})")

fig_cnn_roc.add_scatter(
    x=[0, 1], y=[0, 1],
    mode="lines",
    line=dict(dash="dash"),
    name="Random"
)

fig_cnn_roc.update_layout(
    height=500,
    width=800,
    title="ROC Curve - Custom CNN",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate"
)

fig_cnn_roc.show()

joblib.dump(fig_cnn_roc, CHARTS_DIR / "ROC_Curve-Custom_CNN.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\gender\\ROC_Curve-Custom_CNN.pkl']

### ResNet50 Model:

In [72]:
# Train - Female
fpr_tr_female, tpr_tr_female, _ = roc_curve(
    (resnet_train_results["labels"] == 0).astype(int),
    resnet_train_results["probs"][:, 0]
)

# Train - Male
fpr_tr_male, tpr_tr_male, _ = roc_curve(
    (resnet_train_results["labels"] == 1).astype(int),
    resnet_train_results["probs"][:, 1]
)

# Train - Unknown
fpr_tr_unknown, tpr_tr_unknown, _ = roc_curve(
    (resnet_train_results["labels"] == 2).astype(int),
    resnet_train_results["probs"][:, 2]
)

# Validation - Female
fpr_va_female, tpr_va_female, _ = roc_curve(
    (resnet_val_results["labels"] == 0).astype(int),
    resnet_val_results["probs"][:, 0]
)

# Validation - Male
fpr_va_male, tpr_va_male, _ = roc_curve(
    (resnet_val_results["labels"] == 1).astype(int),
    resnet_val_results["probs"][:, 1]
)

# Validation - Unknown
fpr_va_unknown, tpr_va_unknown, _ = roc_curve(
    (resnet_val_results["labels"] == 2).astype(int),
    resnet_val_results["probs"][:, 2]
)

# Test - Female
fpr_te_female, tpr_te_female, _ = roc_curve(
    (resnet_test_results["labels"] == 0).astype(int),
    resnet_test_results["probs"][:, 0]
)

# Test - Male
fpr_te_male, tpr_te_male, _ = roc_curve(
    (resnet_test_results["labels"] == 1).astype(int),
    resnet_test_results["probs"][:, 1]
)

# Test - Unknown
fpr_te_unknown, tpr_te_unknown, _ = roc_curve(
    (resnet_test_results["labels"] == 2).astype(int),
    resnet_test_results["probs"][:, 2]
)

fig_resnet_roc = go.Figure()

fig_resnet_roc.add_scatter(x=fpr_tr_female, y=tpr_tr_female, mode="lines",
    name=f"Train Female (AUC={auc(fpr_tr_female, tpr_tr_female):.3f})")

fig_resnet_roc.add_scatter(x=fpr_tr_male, y=tpr_tr_male, mode="lines",
    name=f"Train Male (AUC={auc(fpr_tr_male, tpr_tr_male):.3f})")

fig_resnet_roc.add_scatter(x=fpr_tr_unknown, y=tpr_tr_unknown, mode="lines",
    name=f"Train Unknown (AUC={auc(fpr_tr_unknown, tpr_tr_unknown):.3f})")

fig_resnet_roc.add_scatter(x=fpr_va_female, y=tpr_va_female, mode="lines",
    name=f"Validation Female (AUC={auc(fpr_va_female, tpr_va_female):.3f})")

fig_resnet_roc.add_scatter(x=fpr_va_male, y=tpr_va_male, mode="lines",
    name=f"Validation Male (AUC={auc(fpr_va_male, tpr_va_male):.3f})")

fig_resnet_roc.add_scatter(x=fpr_va_unknown, y=tpr_va_unknown, mode="lines",
    name=f"Validation Unknown (AUC={auc(fpr_va_unknown, tpr_va_unknown):.3f})")

fig_resnet_roc.add_scatter(x=fpr_te_female, y=tpr_te_female, mode="lines",
    name=f"Test Female (AUC={auc(fpr_te_female, tpr_te_female):.3f})")

fig_resnet_roc.add_scatter(x=fpr_te_male, y=tpr_te_male, mode="lines",
    name=f"Test Male (AUC={auc(fpr_te_male, tpr_te_male):.3f})")

fig_resnet_roc.add_scatter(x=fpr_te_unknown, y=tpr_te_unknown, mode="lines",
    name=f"Test Unknown (AUC={auc(fpr_te_unknown, tpr_te_unknown):.3f})")

fig_resnet_roc.add_scatter(
    x=[0, 1], y=[0, 1],
    mode="lines", line=dict(dash="dash"), name="Random"
)

fig_resnet_roc.update_layout(
    height=500, width=800,
    title="ROC Curve - ResNet50",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate"
)

fig_resnet_roc.show()

joblib.dump(fig_resnet_roc, CHARTS_DIR / "ROC_Curve-ResNet50.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\gender\\ROC_Curve-ResNet50.pkl']

### VGG16 Model:

In [80]:
# Train - Female
fpr_tr_female, tpr_tr_female, _ = roc_curve(
    (vgg_train_results["labels"] == 0).astype(int),
    vgg_train_results["probs"][:, 0]
)

# Train - Male
fpr_tr_male, tpr_tr_male, _ = roc_curve(
    (vgg_train_results["labels"] == 1).astype(int),
    vgg_train_results["probs"][:, 1]
)

# Train - Unknown
fpr_tr_unknown, tpr_tr_unknown, _ = roc_curve(
    (vgg_train_results["labels"] == 2).astype(int),
    vgg_train_results["probs"][:, 2]
)

# Validation - Female
fpr_va_female, tpr_va_female, _ = roc_curve(
    (vgg_val_results["labels"] == 0).astype(int),
    vgg_val_results["probs"][:, 0]
)

# Validation - Male
fpr_va_male, tpr_va_male, _ = roc_curve(
    (vgg_val_results["labels"] == 1).astype(int),
    vgg_val_results["probs"][:, 1]
)

# Validation - Unknown
fpr_va_unknown, tpr_va_unknown, _ = roc_curve(
    (vgg_val_results["labels"] == 2).astype(int),
    vgg_val_results["probs"][:, 2]
)

# Test - Female
fpr_te_female, tpr_te_female, _ = roc_curve(
    (vgg_test_results["labels"] == 0).astype(int),
    vgg_test_results["probs"][:, 0]
)

# Test - Male
fpr_te_male, tpr_te_male, _ = roc_curve(
    (vgg_test_results["labels"] == 1).astype(int),
    vgg_test_results["probs"][:, 1]
)

# Test - Unknown
fpr_te_unknown, tpr_te_unknown, _ = roc_curve(
    (vgg_test_results["labels"] == 2).astype(int),
    vgg_test_results["probs"][:, 2]
)

fig_vgg_roc = go.Figure()

fig_vgg_roc.add_scatter(x=fpr_tr_female, y=tpr_tr_female, mode="lines",
    name=f"Train Female (AUC={auc(fpr_tr_female, tpr_tr_female):.3f})")

fig_vgg_roc.add_scatter(x=fpr_tr_male, y=tpr_tr_male, mode="lines",
    name=f"Train Male (AUC={auc(fpr_tr_male, tpr_tr_male):.3f})")

fig_vgg_roc.add_scatter(x=fpr_tr_unknown, y=tpr_tr_unknown, mode="lines",
    name=f"Train Unknown (AUC={auc(fpr_tr_unknown, tpr_tr_unknown):.3f})")

fig_vgg_roc.add_scatter(x=fpr_va_female, y=tpr_va_female, mode="lines",
    name=f"Validation Female (AUC={auc(fpr_va_female, tpr_va_female):.3f})")

fig_vgg_roc.add_scatter(x=fpr_va_male, y=tpr_va_male, mode="lines",
    name=f"Validation Male (AUC={auc(fpr_va_male, tpr_va_male):.3f})")

fig_vgg_roc.add_scatter(x=fpr_va_unknown, y=tpr_va_unknown, mode="lines",
    name=f"Validation Unknown (AUC={auc(fpr_va_unknown, tpr_va_unknown):.3f})")

fig_vgg_roc.add_scatter(x=fpr_te_female, y=tpr_te_female, mode="lines",
    name=f"Test Female (AUC={auc(fpr_te_female, tpr_te_female):.3f})")

fig_vgg_roc.add_scatter(x=fpr_te_male, y=tpr_te_male, mode="lines",
    name=f"Test Male (AUC={auc(fpr_te_male, tpr_te_male):.3f})")

fig_vgg_roc.add_scatter(x=fpr_te_unknown, y=tpr_te_unknown, mode="lines",
    name=f"Test Unknown (AUC={auc(fpr_te_unknown, tpr_te_unknown):.3f})")

fig_vgg_roc.add_scatter(
    x=[0, 1], y=[0, 1],
    mode="lines", line=dict(dash="dash"), name="Random"
)

fig_vgg_roc.update_layout(
    height=500, width=800,
    title="ROC Curve - VGG16",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate"
)

fig_vgg_roc.show()

joblib.dump(fig_vgg_roc, CHARTS_DIR / "ROC_Curve-VGG_16.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\gender\\ROC_Curve-VGG_16.pkl']

### EfficientNet_B0 Model:

In [75]:
# Train - Female
fpr_tr_female, tpr_tr_female, _ = roc_curve(
    (effnet_train_results["labels"] == 0).astype(int),
    effnet_train_results["probs"][:, 0]
)

# Train - Male
fpr_tr_male, tpr_tr_male, _ = roc_curve(
    (effnet_train_results["labels"] == 1).astype(int),
    effnet_train_results["probs"][:, 1]
)

# Train - Unknown
fpr_tr_unknown, tpr_tr_unknown, _ = roc_curve(
    (effnet_train_results["labels"] == 2).astype(int),
    effnet_train_results["probs"][:, 2]
)

# Validation - Female
fpr_va_female, tpr_va_female, _ = roc_curve(
    (effnet_val_results["labels"] == 0).astype(int),
    effnet_val_results["probs"][:, 0]
)

# Validation - Male
fpr_va_male, tpr_va_male, _ = roc_curve(
    (effnet_val_results["labels"] == 1).astype(int),
    effnet_val_results["probs"][:, 1]
)

# Validation - Unknown
fpr_va_unknown, tpr_va_unknown, _ = roc_curve(
    (effnet_val_results["labels"] == 2).astype(int),
    effnet_val_results["probs"][:, 2]
)

# Test - Female
fpr_te_female, tpr_te_female, _ = roc_curve(
    (effnet_test_results["labels"] == 0).astype(int),
    effnet_test_results["probs"][:, 0]
)

# Test - Male
fpr_te_male, tpr_te_male, _ = roc_curve(
    (effnet_test_results["labels"] == 1).astype(int),
    effnet_test_results["probs"][:, 1]
)

# Test - Unknown
fpr_te_unknown, tpr_te_unknown, _ = roc_curve(
    (effnet_test_results["labels"] == 2).astype(int),
    effnet_test_results["probs"][:, 2]
)

fig_effnet_roc = go.Figure()

fig_effnet_roc.add_scatter(x=fpr_tr_female, y=tpr_tr_female, mode="lines",
    name=f"Train Female (AUC={auc(fpr_tr_female, tpr_tr_female):.3f})")

fig_effnet_roc.add_scatter(x=fpr_tr_male, y=tpr_tr_male, mode="lines",
    name=f"Train Male (AUC={auc(fpr_tr_male, tpr_tr_male):.3f})")

fig_effnet_roc.add_scatter(x=fpr_tr_unknown, y=tpr_tr_unknown, mode="lines",
    name=f"Train Unknown (AUC={auc(fpr_tr_unknown, tpr_tr_unknown):.3f})")

fig_effnet_roc.add_scatter(x=fpr_va_female, y=tpr_va_female, mode="lines",
    name=f"Validation Female (AUC={auc(fpr_va_female, tpr_va_female):.3f})")

fig_effnet_roc.add_scatter(x=fpr_va_male, y=tpr_va_male, mode="lines",
    name=f"Validation Male (AUC={auc(fpr_va_male, tpr_va_male):.3f})")

fig_effnet_roc.add_scatter(x=fpr_va_unknown, y=tpr_va_unknown, mode="lines",
    name=f"Validation Unknown (AUC={auc(fpr_va_unknown, tpr_va_unknown):.3f})")

fig_effnet_roc.add_scatter(x=fpr_te_female, y=tpr_te_female, mode="lines",
    name=f"Test Female (AUC={auc(fpr_te_female, tpr_te_female):.3f})")

fig_effnet_roc.add_scatter(x=fpr_te_male, y=tpr_te_male, mode="lines",
    name=f"Test Male (AUC={auc(fpr_te_male, tpr_te_male):.3f})")

fig_effnet_roc.add_scatter(x=fpr_te_unknown, y=tpr_te_unknown, mode="lines",
    name=f"Test Unknown (AUC={auc(fpr_te_unknown, tpr_te_unknown):.3f})")

fig_effnet_roc.add_scatter(
    x=[0, 1], y=[0, 1],
    mode="lines", line=dict(dash="dash"), name="Random"
)

fig_effnet_roc.update_layout(
    height=500, width=800,
    title="ROC Curve - EfficientNet-B0",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate"
)

fig_effnet_roc.show()

joblib.dump(fig_effnet_roc, CHARTS_DIR / "ROC_Curve-EffNet-B0.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\gender\\ROC_Curve-EffNet-B0.pkl']

### ViT_B16 Model:

In [77]:
# Train - Female
fpr_tr_female, tpr_tr_female, _ = roc_curve(
    (vit_train_results["labels"] == 0).astype(int),
    vit_train_results["probs"][:, 0]
)

# Train - Male
fpr_tr_male, tpr_tr_male, _ = roc_curve(
    (vit_train_results["labels"] == 1).astype(int),
    vit_train_results["probs"][:, 1]
)

# Train - Unknown
fpr_tr_unknown, tpr_tr_unknown, _ = roc_curve(
    (vit_train_results["labels"] == 2).astype(int),
    vit_train_results["probs"][:, 2]
)

# Validation - Female
fpr_va_female, tpr_va_female, _ = roc_curve(
    (vit_val_results["labels"] == 0).astype(int),
    vit_val_results["probs"][:, 0]
)

# Validation - Male
fpr_va_male, tpr_va_male, _ = roc_curve(
    (vit_val_results["labels"] == 1).astype(int),
    vit_val_results["probs"][:, 1]
)

# Validation - Unknown
fpr_va_unknown, tpr_va_unknown, _ = roc_curve(
    (vit_val_results["labels"] == 2).astype(int),
    vit_val_results["probs"][:, 2]
)

# Test - Female
fpr_te_female, tpr_te_female, _ = roc_curve(
    (vit_test_results["labels"] == 0).astype(int),
    vit_test_results["probs"][:, 0]
)

# Test - Male
fpr_te_male, tpr_te_male, _ = roc_curve(
    (vit_test_results["labels"] == 1).astype(int),
    vit_test_results["probs"][:, 1]
)

# Test - Unknown
fpr_te_unknown, tpr_te_unknown, _ = roc_curve(
    (vit_test_results["labels"] == 2).astype(int),
    vit_test_results["probs"][:, 2]
)

fig_vit_roc = go.Figure()

fig_vit_roc.add_scatter(x=fpr_tr_female, y=tpr_tr_female, mode="lines",
    name=f"Train Female (AUC={auc(fpr_tr_female, tpr_tr_female):.3f})")

fig_vit_roc.add_scatter(x=fpr_tr_male, y=tpr_tr_male, mode="lines",
    name=f"Train Male (AUC={auc(fpr_tr_male, tpr_tr_male):.3f})")

fig_vit_roc.add_scatter(x=fpr_tr_unknown, y=tpr_tr_unknown, mode="lines",
    name=f"Train Unknown (AUC={auc(fpr_tr_unknown, tpr_tr_unknown):.3f})")

fig_vit_roc.add_scatter(x=fpr_va_female, y=tpr_va_female, mode="lines",
    name=f"Validation Female (AUC={auc(fpr_va_female, tpr_va_female):.3f})")

fig_vit_roc.add_scatter(x=fpr_va_male, y=tpr_va_male, mode="lines",
    name=f"Validation Male (AUC={auc(fpr_va_male, tpr_va_male):.3f})")

fig_vit_roc.add_scatter(x=fpr_va_unknown, y=tpr_va_unknown, mode="lines",
    name=f"Validation Unknown (AUC={auc(fpr_va_unknown, tpr_va_unknown):.3f})")

fig_vit_roc.add_scatter(x=fpr_te_female, y=tpr_te_female, mode="lines",
    name=f"Test Female (AUC={auc(fpr_te_female, tpr_te_female):.3f})")

fig_vit_roc.add_scatter(x=fpr_te_male, y=tpr_te_male, mode="lines",
    name=f"Test Male (AUC={auc(fpr_te_male, tpr_te_male):.3f})")

fig_vit_roc.add_scatter(x=fpr_te_unknown, y=tpr_te_unknown, mode="lines",
    name=f"Test Unknown (AUC={auc(fpr_te_unknown, tpr_te_unknown):.3f})")

fig_vit_roc.add_scatter(
    x=[0, 1], y=[0, 1],
    mode="lines", line=dict(dash="dash"), name="Random"
)

fig_vit_roc.update_layout(
    height=500, width=800,
    title="ROC Curve - ViT-B16",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate"
)

fig_vit_roc.show()

joblib.dump(fig_vit_roc, CHARTS_DIR / "ROC_Curve-ViT_B16.pkl")

['C:\\Users\\jagad\\Documents\\my_classes\\Tasks\\my_projects\\Deep_Fake_project_final\\charts\\gender\\ROC_Curve-ViT_B16.pkl']

# Compare the models

The best model is selected using validation macro F1-score, not test performance. This keeps the test set reserved for final evaluation.

In [79]:
rows = []
for name, var in model_list:
    if var + "_test_results" in globals():
        val_r = globals()[var + "_val_results"]
        test_r = globals()[var + "_test_results"]
        rows.append({"Model": name,
                     "Val F1": round(val_r["f1"], 3),
                     "Test Accuracy": round(test_r["accuracy"], 3),
                     "Test F1": round(test_r["f1"], 3),
                     "Test AUC": round(test_r["auc"], 3)})

comparison = pd.DataFrame(rows).sort_values("Val F1", ascending=False).reset_index(drop=True)
comparison.to_csv(MODEL_DIR / "results.csv", index=False)
best_model_name = comparison.loc[0, "Model"]
print("Best model for", TASK, ":", best_model_name)
comparison

Best model for gender : ViT_B16


,Model,Val F1,Test Accuracy,Test F1,Test AUC
0,ViT_B16,0.912,0.933,0.915,0.979
1,ResNet50,0.911,0.915,0.888,0.983
2,VGG16,0.899,0.918,0.892,0.972
3,EfficientNet_B0,0.881,0.900,0.868,0.976
4,CNN,0.748,0.778,0.707,0.908
